# Homework: two real questions, worked solution

In [ ]:
import csv
import os
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)


def load_rows(path):
    """Return the CSV at path as a list of dictionaries."""
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


rows = load_rows("data/clean/plays.csv")
len(rows)

## Question 1: average minutes per genre

The two-dictionary shape: one counts the plays, one sums the minutes, and
the answer is one divided by the other.

In [ ]:
def average_minutes_by_genre(rows):
    """Return a dictionary of genre -> average minutes per play."""
    plays = {}
    minutes = {}

    for row in rows:
        genre = row["genre"]
        plays[genre] = plays.get(genre, 0) + 1
        minutes[genre] = minutes.get(genre, 0.0) + float(row["minutes_played"])

    return {genre: minutes[genre] / plays[genre] for genre in plays}


def top(counts, n=5):
    """Return the n highest (key, value) pairs, biggest first."""
    return sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:n]


averages = average_minutes_by_genre(rows)

print("Average minutes per play, by genre:\n")
for genre, mean in top(averages, n=len(averages)):
    print(f"  {genre:12} {mean:>6.2f}")

winner, best = top(averages, n=1)[0]
print(f"\n{winner} has the longest average play, at {best:.2f} minutes.")

### The trap in question 1

Electronic has by far the most plays **and** the most total minutes, so if
you answer with a sum instead of an average you get Electronic and it looks
convincing.

On the average, Jazz wins comfortably: far fewer plays, but much longer ones.

In [ ]:
totals = {}
for row in rows:
    totals[row["genre"]] = totals.get(row["genre"], 0.0) + float(row["minutes_played"])

print("by TOTAL minutes:  ", top(totals, 3))
print("by AVERAGE minutes:", [(g, round(m, 2)) for g, m in top(averages, 3)])

"Highest total" and "highest average" are different questions, and reaching
for the wrong one is the most common way to produce a confident wrong answer
in data work.

## Question 2: most played artist in 2025

In [ ]:
def plays_by_artist(rows, year=None):
    """Return a dictionary of artist -> play count, optionally for one year."""
    counts = {}
    for row in rows:
        if year is not None and not row["played_at"].startswith(str(year)):
            continue
        artist = row["artist_name"]
        counts[artist] = counts.get(artist, 0) + 1
    return counts


counts_2025 = plays_by_artist(rows, year=2025)

print("Most played artists in 2025:\n")
for artist, number in top(counts_2025):
    print(f"  {artist:24} {number:>4}")

### Why printing the top five matters

Look at the top two rows. **Glass Tram and Sara Lindqvist are tied on 115
plays each.**

`sorted()` has to put one of them first, and it picks whichever it happened
to meet first in the data. So "the most played artist of 2025" has an
arbitrary winner, and a program that printed only the top row would have
stated it with total confidence.

In [ ]:
print(top(counts_2025, 1))

When a single answer comes out of a ranking, it is always worth a glance at
second place to see how close it was.

## The stretch: skip rate per device

In [ ]:
def skip_rate_by_device(rows):
    """Return a dictionary of device -> percentage of plays skipped."""
    plays = {}
    skips = {}

    for row in rows:
        device = row["device"]
        plays[device] = plays.get(device, 0) + 1
        if row["skipped"] == "1":
            skips[device] = skips.get(device, 0) + 1

    return {d: 100 * skips.get(d, 0) / plays[d] for d in plays}


for device, rate in top(skip_rate_by_device(rows), n=5):
    print(f"  {device:10} {rate:>5.1f}%")

The speaker gets skipped most and the car least, which makes a certain
sense: reaching for your phone while driving is inconvenient.

That is the kind of small, plausible story a dataset can support. Notice
that it is a **story**, not a finding: nothing here proves why, and the
difference between 14.0% and 10.6% on a few hundred plays is not a lot to
hang an argument on.